Load the dataset with PySpark

In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path
from pyspark.sql import functions as F
import country_converter as coco
from itertools import chain

CLEANED_DATA_DIR = Path("../data/cleaned")
PROCESSED_DATA_DIR = Path("../data")

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet(f"{PROCESSED_DATA_DIR}/4_eea_co2_emissions_from_passenger_cars-001.parquet")

last_present = df.count()
print(f"{last_present} rows present")

Remap column names to more easily understandable ones

In [ ]:
column_mapping = {
    "Country": "geo",
    "VFN": "vehicle_family_id_number",
    "Mh": "manufacturer_name_eu_standard_denomination",
    "T": "type",
    "Va": "variant",
    "Ve": "version",
    "Mk": "make",
    "Cn": "commercial_name",
    "m (kg)": "mass_in_running_order (kg)",
    "Ewltp (g/km)": "co2_emissions_WLTP (g/km)",
    "Ft": "Motor energy",
    "Fm": "fuel_mode",
    "ec (cm3)": "engine_capacity (cm3)",
    "ep (KW)": "engine_power (KW)",
    "z (Wh/km)": "electric_energy_consumption (Wh/km)",
    "r": "registrations",
    "year": "TIME_PERIOD"
}

for old_col, new_col in column_mapping.items():
    df = df.withColumnRenamed(old_col, new_col)

Drop non EU countries

In [ ]:
eu27_2020 = ['BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV', 'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 'PT', 'RO', 'SI', 'SK', 'FI', 'SE']
country_dict = {code: coco.convert(names=code, to='name_short', not_found=None) for code in eu27_2020}

df = df.filter(df["geo"].isin(eu27_2020))

last_present = df.count()

Drop  N/A values

In [ ]:
num_cols = [
   "mass_in_running_order (kg)", "co2_emissions_WLTP (g/km)", 
    "engine_capacity (cm3)", "engine_power (KW)", "electric_energy_consumption (Wh/km)"
]
str_cols = ["geo", "commercial_name", "Motor energy"]
all_cols = str_cols + num_cols

df = df.na.drop(subset=str_cols)
df = df.filter(df["TIME_PERIOD"] != 0)

print(f"Row count changed by: {last_present - df.count()}")
last_present = df.count()
print(f"{last_present} rows present")

Drop columns we don't need

In [ ]:
not_needed = ["vehicle_family_id_number", "version", "make", "fuel_mode", "type", "variant"]
df = df.drop(*not_needed)

Clean the Motor Energy column to remove all various duplicates and unnecessary values, then group the various motor energy types in the ones we intend to analyse

In [ ]:
df = df.withColumn("Motor energy", 
    F.regexp_replace(F.trim(F.lower(F.col("Motor energy"))), "-", "/")
)

unnecessary = ["unknown", "other"]
df = df.filter(~df["Motor energy"].isin(unnecessary))

df = df.withColumn(
    "Motor energy",
    F.when(F.col("Motor energy") == "electric", "Electricity")
     .when(F.col("Motor energy") == "petrol phev", "Petrol plug-in Hybrid")
     .when(F.col("Motor energy").isin("petrol/electric", "hybrid/petrol/e"), "Petrol hybrid")
     .when(F.col("Motor energy") == "diesel/electric", "Diesel hybrid")
     .when(F.col("Motor energy") == "petrol", "Petrol (excluding hybrids)")
     .when(F.col("Motor energy") == "diesel", "Diesel (excluding hybrids)")
     .otherwise("Alternative/Other") 
)
print(f"Row count changed by: {last_present - df.count()}")
last_present = df.count()
print(f"{last_present} rows present")

Clean the manufacturer_name_eu_standard_denomination column, removing unnecessary values and reducing duplicates

In [ ]:
df = df.filter(~F.col("manufacturer_name_eu_standard_denomination").isin("DUPLICATE", "OUT OF SCOPE", "UNKNOWN", "duplicate", "unknown"))

name_map = {
    "AUDI HUNGARIA": "AUDI AG",
    "AUDI SPORT": "AUDI AG",
    "BEE": "BEE BEE",
    "BLUECAR ITALY": "BLUECAR",
    "BMW GMBH": "BMW AG",
    "DONGFENG LIUZHOU": "DONGFENG",
    "DONGFENG MOTOR": "DONGFENG",
    "DONKEVOORT": "DONKERVOORT",
    "DR MOTOR": "DR AUTOMOBILES",
    "Duplicate": "DUPLICATE",
    "FORD INDIA": "FORD MOTOR COMPANY",
    "FORD MOTOR AUSTRALIA": "FORD MOTOR COMPANY",
    "FORD WERKE GMBH": "FORD MOTOR COMPANY",
    "Ford Motor Company": "FORD MOTOR COMPANY",
    "GENERAL MOTORS COMPANY": "GENERAL MOTORS",
    "GENERAL MOTORS HOLDINGS": "GENERAL MOTORS",
    "GM ITALIA": "GENERAL MOTORS",
    "GM KOREA": "GENERAL MOTORS",
    "GM KOREA": "GENERAL MOTORS",
    "HONDA CHINA": "HONDA",
    "HONDA MOTOR CO": "HONDA",
    "HONDA THAILAND": "HONDA",
    "HONDA TURKIYE": "HONDA",
    "HONDA UK": "HONDA",
    "HYUNDAI ASSAN": "HYUNDAI",
    "HYUNDAI ASSAN": "HYUNDAI",
    "HYUNDAI CZECH": "HYUNDAI",
    "HYUNDAI CZECH": "HYUNDAI",
    "HYUNDAI EUROPE": "HYUNDAI",
    "HYUNDAI INDIA": "HYUNDAI",
    "JIANGXI JIANGLING": "JIANGLING MOTOR",
    "KIA SLOVAKIA": "KIA",
    "KIA SLOVAKIA": "KIA",
    "LADA FRANCE": "LADA",
    "LANZHOU ZHIDOU": "LANZHOU",
    "MAGYAR SUZUKI": "SUZUKI MOTOR CORPORATION",
    "MARUTI SUZUKI": "SUZUKI MOTOR CORPORATION",
    "MAZDA EUROPE": "MAZDA",
    "MERCEDES AMG": "MERCEDES-BENZ AG",
    "MERCEDES-AMG": "MERCEDES-BENZ AG",
    "NISSAN AUTOMOTIVE EUROPE": "NISSAN",
    "OPEL AUTOMOBILE": "OPEL",
    "QUATTRO": "AUDI AG",
    "RADICAL MOTOSPORT": "RADICAL MOTORSPORT",
    "ROLLS-ROYCE": "ROLLS ROYCE",
    "SAIC MAXUS": "SAIC MOTOR CORPORATION",
    "SAIC MOTOR": "SAIC MOTOR CORPORATION",
    "STELLANTIS EUROPE": "STELLANTIS AUTO",
    "SUZUKI THAILAND": "SUZUKI MOTOR CORPORATION",
    "TOYOTA MOTOR CORPORATION": "TOYOTA",
    "WUHAN LOTUS": "LOTUS"
}

df = df.replace(name_map, subset=["manufacturer_name_eu_standard_denomination"])
print(f"Row count changed by: {last_present - df.count()}")
last_present = df.count()
print(f"{last_present} rows present")

Aggregate registrations by merging rows that are now duplicated, construct a new standard geopoliticaly entity column like with the other datasets, then finally reorder columns and write the results to disk

In [ ]:
grouping_columns = [col for col in df.columns if col != "registrations"]
df = df.groupBy(*grouping_columns) \
               .agg(F.sum("registrations").alias("registrations"))

mapping_expr = F.create_map([F.lit(x) for x in chain(*country_dict.items())])
df = df.withColumn("Geopolitical entity (reporting)", mapping_expr[F.col("geo")])

df = df.withColumn("registrations", F.col("registrations").cast("integer"))

df = df.select(
    "geo",
    "Geopolitical entity (reporting)",
    "TIME_PERIOD",
    "registrations",
    "manufacturer_name_eu_standard_denomination",
    "commercial_name",
    "Motor energy",
    "mass_in_running_order (kg)",
    "co2_emissions_WLTP (g/km)",
    "engine_capacity (cm3)",
    "engine_power (KW)",
    "electric_energy_consumption (Wh/km)"
)

print(f"Row count changed by: {last_present - df.count()}")
last_present = df.count()
print(f"{last_present} rows present")

df.write.mode("overwrite") \
    .option("compression", "gzip") \
    .parquet(f"{CLEANED_DATA_DIR}/4c_eea_co2_emissions_from_passenger_cars-001.parquet")